# Colab training — UAV Multi-Priority Resource Allocation

Trains **SAC** (`N_CRITICS = 2`, twin critics) or **REDQ-SAC**
(`N_CRITICS = 10`, ensemble critics) on the UAV resource-allocation
environment. Checkpoints live on Google Drive, so a disconnected session can
resume where it stopped.

**Before running**

1. `Runtime -> Change runtime type -> T4 GPU` (CPU works, but is much slower).
2. If the repository is **private**, add a Colab secret named `GITHUB_TOKEN`
   (key icon in the left sidebar) holding a GitHub token with `repo` read
   access. A public repository needs no token.
3. Run the cells top to bottom; only the **configuration** cell needs editing.

**What survives a disconnect**

| Item | Location | Persists |
|------|----------|----------|
| model + optimizer checkpoints | `MyDrive/uavrl/ckpt/<run>` | yes |
| TensorBoard logs / metrics | copied at the end | yes |
| replay buffer | Colab RAM | **no** |

After a disconnect, rerun every cell. Training resumes from the latest
checkpoint and restores model, optimizer and step counter; the replay buffer
is rebuilt from scratch (documented limitation, see the final cell).

Expected runtime for 300k steps on a T4: roughly 30-60 min, depending on GPU
allocation and the evaluation episodes.


In [ ]:
import platform
import sys

import torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Device count:", torch.cuda.device_count())
else:
    print("No GPU detected: training will run on CPU (slower, but supported).")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# ---------------------------------------------------------------------------
# Edit these values only.
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/tra134/Multi-Priority-Resource-Allocation-in-UAV-Networks-research.git"
BRANCH = "main"
WORKDIR = "/content/uav-drl-llm"

SEED = 0
N_CRITICS = 10        # 2 = Pure SAC (twin critics), 10 = REDQ-SAC
STEPS = 300_000
CKPT_EVERY = 5_000    # shorter interval = less work lost on a disconnect
EVAL_EVERY = 20_000
EVAL_EPISODES = 20

DRIVE_ROOT = "/content/drive/MyDrive/uavrl"
RUN_NAME = f"sac_seed{SEED}_N{N_CRITICS}"   # must match scripts/train.py

CKPT_DIR = f"{DRIVE_ROOT}/ckpt/{RUN_NAME}"
TB_DIR = f"{WORKDIR}/results/tb/{RUN_NAME}"          # local = faster
RESULTS_DIR = f"{WORKDIR}/results/{RUN_NAME}"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/{RUN_NAME}"

print("Run          :", RUN_NAME)
print("Checkpoints  :", CKPT_DIR)
print("TensorBoard  :", TB_DIR)
print("Drive results:", DRIVE_RESULTS)


In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)


# Make the checkout importable even if the editable install was skipped or
# was installed into a different interpreter.
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    [os.path.join(WORKDIR, "src"), env.get("PYTHONPATH", "")]
).strip(os.pathsep)

# Optional GitHub token from Colab Secrets; only needed for private repos.
token = ""
try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass

clone_url = REPO_URL
if token:
    clone_url = REPO_URL.replace("https://", f"https://{token}@", 1)

if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    # capture_output keeps the token-bearing URL out of the notebook output.
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, clone_url, WORKDIR],
        check=True,
        capture_output=True,
        text=True,
    )
    print("cloned", REPO_URL)
else:
    # Fast-forward only: refuses to overwrite local edits in the workspace.
    subprocess.run(
        ["git", "-C", WORKDIR, "fetch", "origin", BRANCH],
        check=True,
        capture_output=True,
        text=True,
    )
    subprocess.run(["git", "-C", WORKDIR, "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", WORKDIR, "pull", "--ff-only", "origin", BRANCH], check=True
    )
    print("updated", REPO_URL)

# Remove the token from .git/config now that the clone/fetch is done.
run(["git", "remote", "set-url", "origin", REPO_URL], cwd=WORKDIR)
run(["git", "log", "--oneline", "-3"], cwd=WORKDIR)
run(["git", "rev-parse", "HEAD"], cwd=WORKDIR)

# Install with the SAME interpreter that runs this notebook (not a bare
# "python", which may point at a different environment in Colab).
run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=WORKDIR, env=env)

check = subprocess.run(
    [sys.executable, "-c", "import uavrl, uavrl.env.uav_env; print(uavrl.__file__)"],
    cwd=WORKDIR,
    env=env,
    capture_output=True,
    text=True,
)
if check.returncode != 0:
    raise RuntimeError("uavrl is not importable after install:\n" + check.stderr)
print("uavrl import OK:", check.stdout.strip())


In [ ]:
import os
import subprocess
import sys

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(TB_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Same guarantee as the install cell: the source tree is always on sys.path.
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    [os.path.join(WORKDIR, "src"), env.get("PYTHONPATH", "")]
).strip(os.pathsep)


def import_check():
    return subprocess.run(
        [sys.executable, "-c", "import uavrl, uavrl.env.uav_env"],
        cwd=WORKDIR,
        env=env,
        capture_output=True,
        text=True,
    )


# Self-heal when the runtime restarted and the install cell was skipped.
check = import_check()
if check.returncode != 0:
    print("uavrl is not importable; installing the current checkout...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
        cwd=WORKDIR,
        env=env,
        check=True,
    )
    check = import_check()
if check.returncode != 0:
    raise RuntimeError("uavrl is still not importable:\n" + check.stderr)

cmd = [
    sys.executable, "scripts/train.py",
    "--config", "configs/default.yaml",
    "--agent", "sac",
    "--n-critics", str(N_CRITICS),
    "--seed", str(SEED),
    "--steps", str(STEPS),
    "--ckpt-dir", CKPT_DIR,
    "--ckpt-every", str(CKPT_EVERY),
    "--eval-every", str(EVAL_EVERY),
    "--eval-episodes", str(EVAL_EPISODES),
    "--tb-dir", TB_DIR,
    "--resume",
]
print("interpreter:", sys.executable)
print("$", " ".join(cmd))
try:
    subprocess.run(cmd, cwd=WORKDIR, env=env, check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        f"Training failed with exit code {exc.returncode}. "
        "Read the Python traceback printed above this error. Import errors "
        "usually mean the clone/install cell was skipped after a runtime "
        "restart; rerun it and then run this cell again."
    ) from exc


In [ ]:
import os

import torch

latest = os.path.join(CKPT_DIR, f"{RUN_NAME}_latest.pt")
checkpoint = torch.load(latest, map_location="cpu", weights_only=False)
print("checkpoint :", latest)
print("step       :", checkpoint.get("step"))
print("total_it   :", checkpoint.get("total_it"))
print("resumable  :", "actor_optimizer" in checkpoint)
print("critics    :", len({
    key.split(".")[1]
    for key in checkpoint["critic"]
    if key.startswith("critics.")
}))


In [ ]:
import os
import shutil

os.makedirs(DRIVE_RESULTS, exist_ok=True)

# TensorBoard logs and local results -> Drive.
if os.path.isdir(TB_DIR):
    shutil.copytree(TB_DIR, os.path.join(DRIVE_RESULTS, "tb"), dirs_exist_ok=True)
if os.path.isdir(RESULTS_DIR):
    shutil.copytree(
        RESULTS_DIR, os.path.join(DRIVE_RESULTS, "results"), dirs_exist_ok=True
    )

# Checkpoints already live on Drive; report the newest ones.
ckpts = sorted(f for f in os.listdir(CKPT_DIR) if f.endswith(".pt"))
print(f"{len(ckpts)} checkpoint(s) in {CKPT_DIR}")
for name in ckpts[-5:]:
    size_mb = os.path.getsize(os.path.join(CKPT_DIR, name)) / 1e6
    print(f"  {name}  ({size_mb:.1f} MB)")
print("results copied to", DRIVE_RESULTS)


In [ ]:
# Optional: live TensorBoard (run after the training cell has produced logs).
%load_ext tensorboard
%tensorboard --logdir {TB_DIR}


## After training

**Checkpoints** are in `MyDrive/uavrl/ckpt/<run>/`:

- `sac_seed0_N10_latest.pt` — newest state, used by `--resume`
- `sac_seed0_N10_step<k>.pt` — periodic snapshots

**Run both agents (Phase 3 + Phase 4):** rerun the notebook twice, changing only
`N_CRITICS` in the configuration cell (`2` = Pure SAC, `10` = REDQ-SAC). The run
name changes automatically, so checkpoints never overwrite each other.

**Resume limitation:** a checkpoint restores the actor, critics, optimizers,
entropy temperature and global step. The replay buffer is not saved, so after a
disconnect the first updates wait until the buffer refills to one batch. For
short Colab sessions keep `CKPT_EVERY` at `5000` or lower.

**Next phases:** download the checkpoint for `tau_unc` calibration and the
Phase 9 comparison grid. The trained policies are also what the LLM hybrid
controller needs (`scripts/llm_controller_smoke.py --ckpt ...`).
